In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from faker import Faker
import random

fake = Faker('id_ID') #clone of indonesian locale

# target number of rows
n_records = 1234

package_type = ['solo', 'family', 'honeymoon', 'group']
months = pd.date_range(start='2023-01-01', periods=12, freq='M').strftime('%Y-%m').tolist()
gender = ['male', 'female', 'other']
customer_type = ['regular', 'new']


# function to generate random email based on name
def generate_email(name):
    name = name
    domains =  ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com']
    domain = random.choice(domains)
    return f"{name}@{domain}"

# Generate random data - 1st table: customer data
customer_names = [fake.name() for _ in range(n_records)]

# id, name, gender, age, nationality, email, customer_type
customer_data = {
    'customer_id': [f"MY{1000 + i}" for i in range(n_records)],
    'customer_name': customer_names,
    'gender': [random.choice(gender) for _ in range(n_records)],
    'age': [fake.random_int(min=18, max=70) for _ in range(n_records)],
    'email': [generate_email(name) for name in customer_names],
    'customer_type': [random.choice(customer_type) for _ in range(n_records)]    
}

customer = pd.DataFrame(customer_data)
customer.head()

C:\Users\User\AppData\Local\Temp\ipykernel_12284\4192148590.py:13: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  months = pd.date_range(start='2023-01-01', periods=12, freq='M').strftime('%Y-%m').tolist()


,customer_id,customer_name,gender,age,email,customer_type
0,MY1000,Ghaliyati Kurniawan,female,31,Ghaliyati Kurniawan@hotmail.com,regular
1,MY1001,Kasusra Prasetyo,male,23,Kasusra Prasetyo@outlook.com,regular
2,MY1002,Gada Saefullah,male,56,Gada Saefullah@outlook.com,regular
3,MY1003,R. Kanda Kuswoyo,male,20,R. Kanda Kuswoyo@gmail.com,regular
4,MY1004,Iriana Lazuardi,other,40,Iriana Lazuardi@gmail.com,regular


In [84]:
# Cleaned customer data
customer['email'] = customer['email'].str.replace(' ', '_').str.lower()
customer['email']

0         ghaliyati_kurniawan@hotmail.com
1            kasusra_prasetyo@outlook.com
2              gada_saefullah@outlook.com
3              r._kanda_kuswoyo@gmail.com
4               iriana_lazuardi@gmail.com
                      ...                
1229          farhunnisa_kusumo@gmail.com
1230             empluk_hastuti@yahoo.com
1231         maryadi_pangestu@outlook.com
1232            faizah_yulianti@gmail.com
1233    mustofa_budiyanto,_s.pt@yahoo.com
Name: email, Length: 1234, dtype: object

In [85]:
# Generate random data - 2nd table: travle_package
# col name: package_id, package_name, package_type, duration_dayys, price_per_pax, category

destination_malaysia = ['Kuala Lumpur', 'Penang', 'Melaka', 'Langkawi', 'Pulau Perhentian', 'Pulau Redang', 'Pulau Tioman', 'Pulau Sipadan', 'Cameron Highlands', 'Kota Kinabalu', 'Taman Negara', 'Borneo', 'Putrajaya'] 
southeast_asia_destinations = [
    "Bali","Jakarta","Yogyakarta","Bangkok","Chiang Mai","Phuket","Krabi","Hanoi","Ho Chi Minh City","Halong Bay","Sapa","Nha Trang","Luang Prabang","Vientiane","Phnom Penh","Siem Reap","Bagan","Yangon",              
    "Singapore","Manila","Cebu","Boracay","Brunei","Dili","Lombok","Komodo Island" ]

all_destinations = destination_malaysia + southeast_asia_destinations
n_package = len(all_destinations)

# Function to determine base price_per_pax for each package based on type and destination
def get_price(name, type):
    if name in destination_malaysia:
        if type == 'family':
            return fake.random_int(500, 800)
        elif type == 'group': 
            return fake.random_int(800, 1000)
        elif type == 'honeymoon':
            return fake.random_int(1000, 2000)
        elif type == 'solo':
            return fake.random_int(800, 1200)
    elif name in southeast_asia_destinations:
        if type == 'family':
            return fake.random_int(1500, 1900)
        elif type == 'group':
            return fake.random_int(1700, 2700)
        elif type == 'honeymoon':
            return fake.random_int(2500, 3500)
        elif type == 'solo':
            return fake.random_int(1700, 3000)

package_name = [random.choice(all_destinations) for _ in range(n_package)]
package_type = [random.choice(package_type) for _ in range(n_package)]

price_per_pax = [get_price(name, type) for name, type in zip(package_name, package_type)]

travel = {
    'package_id': [f"PC{10 + i}" for i in range(n_package)],
    'package_name': package_name,
    'package_type': package_type,
    'duration_days': [fake.random_int(min=2, max=7) for _ in range(n_package)],
    'price_per_pax': price_per_pax,
    'category': [random.choice(['beach', 'adventure', 'cultural', 'eco']) for _ in range(n_package)]
}

travel_package = pd.DataFrame(travel)
travel_package.head()

,package_id,package_name,package_type,duration_days,price_per_pax,category
0,PC10,Pulau Tioman,group,5,896,eco
1,PC11,Boracay,solo,4,2083,eco
2,PC12,Dili,family,2,1578,adventure
3,PC13,Singapore,honeymoon,3,3417,beach
4,PC14,Putrajaya,honeymoon,2,1528,beach


In [ ]:
# Generate random data - 3rd table: sales

# listing customer_id and package_id for later use
customer_ids = list(customer['customer_id'])
package_ids = list(travel_package['package_id'])

# Create mapping: package_id → package_type
package_type_map = travel_package.set_index('package_id')['package_type'].to_dict()

# Function to determine num_of_pax
def get_pax(package_type):
    ptype = package_type.lower()
    if ptype == 'family':
        return fake.random_int(3, 7)
    elif ptype == 'honeymoon':
        return 2
    elif ptype == 'group':
        return fake.random_int(12, 25)
    else:
        return 1  # fallback

# Number of records to generate
total_sales_record = 2561

# Randomly assign customer_id and package_id
booking_customer_ids = [random.choice(customer_ids) for _ in range(total_sales_record)]
booking_package_ids = [random.choice(package_ids) for _ in range(total_sales_record)]

# Use package_id to get package_type and then determine pax
booking_package_types = [package_type_map[pid] for pid in booking_package_ids]
num_of_pax = [get_pax(ptype) for ptype in booking_package_types]

# Generate the full booking dataset
booking_data = {
    'booking_id': [f"BK{100 + i}" for i in range(total_sales_record)],
    'customer_id': booking_customer_ids,
    'package_id': booking_package_ids,
    'booking_date': [fake.date_between(start_date='-3y', end_date='today') for _ in range(total_sales_record)],
    'travel_date': [fake.date_between(start_date='-1y', end_date='+2y') for _ in range(total_sales_record)],
    'num_of_pax': num_of_pax,
    'discount_promotion': [fake.random_int(min=0, max=17) for _ in range(total_sales_record)],
    'chanel': [random.choice(['online', 'offline', 'agent', 'walk in']) for _ in range(total_sales_record)]
}

# Create DataFrame
sales = pd.DataFrame(booking_data)
sales.head()


,booking_id,customer_id,package_id,booking_date,travel_date,num_of_pax,discount_promotion,chanel
0,BK100,MY2194,PC39,2022-11-07,2027-05-22,5,2,agent
1,BK101,MY1830,PC30,2023-09-26,2027-02-09,19,16,walk in
2,BK102,MY1690,PC23,2024-11-23,2026-02-27,1,13,offline
3,BK103,MY1827,PC22,2023-08-27,2024-12-31,1,9,walk in
4,BK104,MY2099,PC11,2022-09-09,2025-10-06,1,11,agent


In [ ]:
# Generate random data - 4th table: customer_review

booking_ids = sales['booking_id'].to_list()
feedback_comments = [
    "The hotel and guide exceeded our expectations!",
    "Everything was well organized and smooth.",
    "Truly a memorable adventure for our family!",
    "Beautiful destinations and a relaxing itinerary.",
    "Highly recommend this package to others.",
    "Some meals weren’t up to standard.",
    "Overall good, but the hotel could be better.",
    "Tour guide was late and disorganized.",
    "The schedule was a bit tight but manageable.",
    "Too many hidden charges in the package."
]

num_feedbacks = 320

feedback_data = []
for i in range(1, num_feedbacks + 1):
    customer_id = random.choice(customer_ids)
    package_id = random.choice(package_ids)
    booking_id = random.choice(booking_ids)
    satisfaction_score = random.randint(1, 10)
    feedback_date = fake.date_between(start_date='-1y', end_date='today')
    comments = random.choice(feedback_comments)
    rating_hotel = random.randint(2, 5)
    rating_guide = random.randint(2, 5)
    rating_itinerary = random.randint(2, 5)
    would_recommend = random.choice(["Yes", "No"])
    
    feedback_data.append({
            "FeedbackID": i,
            "customer_id": customer_id,
            "package_id": package_id,
            "booking_id": booking_id,
            "SatisfactionScore": satisfaction_score,
            "FeedbackDate": feedback_date,
            "Comments": comments,
            "Rating_Hotel": rating_hotel,
            "Rating_Guide": rating_guide,
            "Rating_Itinerary": rating_itinerary,
            "WouldRecommend": would_recommend
        })

feedback = pd.DataFrame(feedback_data)
feedback.head()


,FeedbackID,customer_id,package_id,booking_id,SatisfactionScore,FeedbackDate,Comments,Rating_Hotel,Rating_Guide,Rating_Itinerary,WouldRecommend
0,1,MY1113,PC11,BK1580,4,2025-05-18,Everything was well organized and smooth.,4,3,5,Yes
1,2,MY1375,PC42,BK2347,7,2024-12-03,The schedule was a bit tight but manageable.,2,5,3,No
2,3,MY1368,PC12,BK1645,9,2025-05-19,Tour guide was late and disorganized.,2,2,5,No
3,4,MY1671,PC43,BK1654,2,2024-09-29,Some meals weren’t up to standard.,2,4,5,No
4,5,MY1948,PC41,BK2325,4,2024-11-30,Truly a memorable adventure for our family!,4,5,3,No


In [102]:
# Generate data - 5th table: Travel_guide
# staff_id, staff_name, role, working_status, commission_rate, 

guide_name = ['Nasrul', 'Khair', 'Intan', 'Amira', 'Nuha', 'Alissa', 'Waddah', 'Izzah', 'Nasir']
role = ['main_guide', 'assistant_guide']
working_status = ['permanent', 'contract', 'freelance']

#generate functions to determine comission rate based on working_status

def get_comission_rate(working_status):
    if working_status == 'freelance':
        return fake.random_int(20, 30) / 100  # 10% to 20%
    elif working_status == 'contract':
        return fake.random_int(10, 17) / 100
    else:
        return fake.random_int(5, 10) / 100
    
staff_data = {
    'staff_id': [str(i).zfill(4) for i in range(101, 101 + len(guide_name))],
    'staff_name': guide_name,
    'role': [random.choice(role) for _ in range(len(guide_name))],
    'working_status': [random.choice(working_status) for _ in guide_name]
}

travel_guide = pd.DataFrame(staff_data)
    
# col comission_rate
travel_guide['comission_rate'] = travel_guide['working_status'].apply(get_comission_rate)
travel_guide.head()




,staff_id,staff_name,role,working_status,comission_rate
0,0101,Nasrul,assistant_guide,contract,0.12
1,0102,Khair,main_guide,contract,0.14
2,0103,Intan,assistant_guide,contract,0.16
3,0104,Amira,assistant_guide,freelance,0.27
4,0105,Nuha,assistant_guide,permanent,0.10


In [108]:
# Generate data - 6th table: Refund
# refund_id, booking_id, refund_date, refund_amount, refund_percentage,reason

# refund reasons
refund_reasons = [
        'Cancellation by customer',
        'Package change request',
        'Travel restrictions due to COVID-19',
        'Unforeseen circumstances',
        'Service quality issues',
        'Booking errors',
        'Natural disasters',
        'Health emergencies',
        'Other']

refund_rows = 9

refund_data = {
    'refund_id': [i for i in range(1, refund_rows + 1)],
    booking_id: [random.choice(sales['booking_id']) for _ in range(refund_rows)],
    'refund_date':[sales['booking_date'] + pd.Timedelta(days=random.randint(1, 30)) for _ in range(refund_rows)],
    'refund_amount': '10 % of package price',
    'refund_percentage': 'based on reason and package type',
    'reason': [random.choice(refund_reasons) for _ in range(refund_rows)]
}

refund = pd.DataFrame(refund_data, index=None)
refund.head()


 

,refund_id,BK948,refund_date,refund_amount,refund_percentage,reason
0,1,BK226,0 2022-12-03 1 2023-10-22 2 ...,10 % of package price,based on reason and package type,Cancellation by customer
1,2,BK2577,0 2022-12-06 1 2023-10-25 2 ...,10 % of package price,based on reason and package type,Service quality issues
2,3,BK2177,0 2022-11-08 1 2023-09-27 2 ...,10 % of package price,based on reason and package type,Other
3,4,BK2033,0 2022-11-21 1 2023-10-10 2 ...,10 % of package price,based on reason and package type,Booking errors
4,5,BK513,0 2022-11-25 1 2023-10-14 2 ...,10 % of package price,based on reason and package type,Other


In [107]:
# Saving all csv files
customer.to_csv(r"C:\Users\User\Desktop\PPNJ\customer.csv", index=False)
travel_package.to_csv(r"C:\Users\User\Desktop\PPNJ\travel_pacakge.csv", index=False)            
sales.to_csv(r"C:\Users\User\Desktop\PPNJ\sales.csv", index=False)
feedback.to_csv(r"C:\Users\User\Desktop\PPNJ\feedback.csv", index=False)
travel_guide.to_csv(r"C:\Users\User\Desktop\PPNJ\travel_guide.csv", index=False)
refund.to_csv(r"C:\Users\User\Desktop\PPNJ\refund.csv", index=False)